# Street-view Affective Rating System

This notebook assigns street-view images to multiple respondents and collects seven Likert-scale affective ratings. It saves progress and exports completed responses.

**Before running:** place non-sensitive test images in `example_images/`, review the respondent and image-count settings in the final lines, and run Jupyter from the repository root. Generated participant data must not be committed to the public repository.

Author: Jingjing Zhao ([ORCID 0009-0002-7522-866X](https://orcid.org/0009-0002-7522-866X))


In [ ]:
import os
import pandas as pd
import random
from datetime import datetime
from IPython.display import display, clear_output
from PIL import Image
import ipywidgets as widgets
import json

class MultiRespondentRatingSystem:
    def __init__(self, image_folder, num_respondents=120, images_per_person=50):
        self.image_folder = image_folder
        self.num_respondents = num_respondents
        self.images_per_person = images_per_person
        self.results_dir = os.path.join(image_folder, "评分结果")
        
        # 创建结果目录
        if not os.path.exists(self.results_dir):
            os.makedirs(self.results_dir)
        
        # 定义评分问题
        self.questions = [
            "1. 街景环境让我觉得时间紧迫—放松",
            "2. 街景环境让我担心来不及—有信心按时赶到",
            "3. 街景环境让我压力很大 – 平静",
            "4. 街景环境让我很累 – 充满活力",
            "5. 街景环境让我无聊 – 兴奋",
            "6. 我认为这次出行是最糟糕的 –是我能想到的最好的",
            "7. 我认为这次出行效果很差 –效果很好"
        ]
        
        # 获取所有图片文件
        self.all_images = [f for f in os.listdir(image_folder) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        # 实验状态变量
        self.current_respondent_id = None
        self.current_respondent_name = ""
        self.current_respondent_images = []
        self.current_image_index = 0
        self.current_scores = [4] * 7
        self.results = pd.DataFrame(columns=['图片名'] + [f'问题{i+1}' for i in range(7)])
        
        # 临时记录文件路径
        self.temp_record_file = os.path.join(self.results_dir, "测试记录临时存储.xlsx")
        
        # 初始化图片使用统计
        self.image_usage = {img: 0 for img in self.all_images}
        
        # 初始化系统
        self.initialize_system()
        self.create_widgets()
    
    def initialize_system(self):
        """初始化系统配置和临时记录"""
        # 如果临时记录文件不存在，创建新的分配方案
        if not os.path.exists(self.temp_record_file):
            self.create_new_assignment_plan()
        else:
            # 加载现有记录
            self.load_existing_records()
    
    def create_new_assignment_plan(self):
        """创建新的图片分配方案"""
        print("创建新的图片分配方案...")
        
        # 重置所有状态
        self.completed_respondents = {}  # 改为字典，存储{受访者ID: 受访者名称}
        self.available_respondents = list(range(1, self.num_respondents + 1))
        random.shuffle(self.available_respondents)  # 随机打乱受访者顺序
        
        # 重置图片使用统计
        self.image_usage = {img: 0 for img in self.all_images}
        self.assignments = {}
        
        for respondent_id in range(1, self.num_respondents + 1):
            # 优先选择使用次数较少的图片
            max_usage = (self.num_respondents * self.images_per_person // len(self.all_images)) + 1
            available_images = [img for img in self.all_images if self.image_usage[img] < max_usage]
            
            if len(available_images) < self.images_per_person:
                available_images = self.all_images.copy()
            
            selected_images = random.sample(available_images, self.images_per_person)
            
            for img in selected_images:
                self.image_usage[img] += 1
            
            self.assignments[respondent_id] = {
                'images': selected_images,
                'respondent_name': f"受访者_{respondent_id}",  # 默认名称
                'status': '待测试'
            }
        
        # 保存分配方案到临时文件
        self.save_records_to_file()
        print(f"新分配方案创建完成！共有{len(self.available_respondents)}个受访者待测试")
    
    def save_records_to_file(self):
        """保存记录到临时Excel文件"""
        # 创建分配详情表
        assignment_data = []
        for respondent_id in range(1, self.num_respondents + 1):
            respondent_data = self.assignments[respondent_id]
            images = respondent_data['images']
            respondent_name = respondent_data['respondent_name']
            status = respondent_data['status']
            
            row_data = {
                '受访者ID': respondent_id, 
                '受访者名称': respondent_name, 
                '状态': status
            }
            for i, img in enumerate(images, 1):
                row_data[f'图片_{i}'] = img
            assignment_data.append(row_data)
        
        assignment_df = pd.DataFrame(assignment_data)
        
        # 创建进度记录表
        completed_count = len([id for id, data in self.assignments.items() if data['status'] == '已完成'])
        available_count = len([id for id, data in self.assignments.items() if data['status'] == '待测试'])
        
        progress_data = {
            '总受访者数': [self.num_respondents],
            '已完成数': [completed_count],
            '待完成数': [available_count],
            '创建时间': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            '最后更新时间': [datetime.now().strftime('%Y-%m-%d %H:%M:%S')]
        }
        progress_df = pd.DataFrame(progress_data)
        
        # 创建使用统计表 - 使用当前的image_usage数据
        usage_stats = pd.DataFrame([
            {'图片文件名': img, '被分配次数': count} 
            for img, count in self.image_usage.items()
        ])
        usage_stats = usage_stats.sort_values('被分配次数', ascending=False)
        
        # 保存到Excel - 明确指定引擎
        try:
            with pd.ExcelWriter(self.temp_record_file, engine='openpyxl') as writer:
                assignment_df.to_excel(writer, sheet_name='分配详情', index=False)
                progress_df.to_excel(writer, sheet_name='进度统计', index=False)
                usage_stats.to_excel(writer, sheet_name='使用统计', index=False)
                
        except Exception as e:
            print(f"保存Excel文件时出错: {e}")
    
    def load_existing_records(self):
        """加载现有记录"""
        try:
            # 明确指定引擎读取Excel文件
            assignment_df = pd.read_excel(self.temp_record_file, sheet_name='分配详情', engine='openpyxl')
            
            # 重置状态
            self.completed_respondents = {}
            self.available_respondents = []
            self.assignments = {}
            
            # 重建分配数据并重新计算图片使用统计
            self.image_usage = {img: 0 for img in self.all_images}
            
            for _, row in assignment_df.iterrows():
                respondent_id = row['受访者ID']
                respondent_name = row['受访者名称']
                status = row['状态']
                
                # 重建图片分配
                images = []
                for i in range(1, self.images_per_person + 1):
                    img_col = f'图片_{i}'
                    if img_col in row and pd.notna(row[img_col]):
                        img_name = row[img_col]
                        images.append(img_name)
                        # 更新图片使用统计
                        if img_name in self.image_usage:
                            self.image_usage[img_name] += 1
                
                self.assignments[respondent_id] = {
                    'images': images,
                    'respondent_name': respondent_name,
                    'status': status
                }
                
                # 检查是否已完成
                if status == '已完成':
                    self.completed_respondents[respondent_id] = respondent_name
                else:
                    self.available_respondents.append(respondent_id)
            
            print(f"加载现有记录: 已完成{len(self.completed_respondents)}个, 待完成{len(self.available_respondents)}个")
            
        except Exception as e:
            print(f"加载记录失败: {e}, 创建新方案")
            self.create_new_assignment_plan()
    
    def get_next_respondent(self):
        """获取下一个待测试的受访者（只使用未测试的数据）"""
        if not self.available_respondents:
            return None, "所有受访者已完成测试"
        
        # 只从待测试的受访者中选择
        available_ids = [id for id in self.available_respondents 
                        if self.assignments[id]['status'] == '待测试']
        
        if not available_ids:
            return None, "所有受访者已完成测试"
        
        next_id = available_ids[0]
        next_name = self.assignments[next_id]['respondent_name']
        return next_id, next_name
    
    def update_respondent_name(self, respondent_id, new_name):
        """更新受访者名称"""
        if respondent_id in self.assignments:
            self.assignments[respondent_id]['respondent_name'] = new_name
            # 更新临时记录文件
            self.save_records_to_file()
    
    def start_next_respondent(self):
        """开始下一个受访者的测试"""
        next_id, next_name = self.get_next_respondent()
        if next_id is None:
            return False
        
        self.start_specific_respondent(next_id, next_name)
        return True
    
    def start_specific_respondent(self, respondent_id, respondent_name):
        """开始特定受访者的评分"""
        if respondent_id not in self.assignments:
            self.status_label.value = f"错误: 找不到受访者 {respondent_name} 的分配数据"
            return
        
        # 检查是否已经完成
        if self.assignments[respondent_id]['status'] == '已完成':
            self.status_label.value = f"受访者 {respondent_name} 已完成测试，请选择其他受访者"
            return
        
        self.current_respondent_id = respondent_id
        self.current_respondent_name = respondent_name
        self.current_respondent_images = self.assignments[respondent_id]['images']
        self.current_image_index = 0
        self.current_scores = [4] * 7
        
        # 检查是否有现有进度
        progress_file = os.path.join(self.results_dir, f'progress_respondent_{respondent_id}.csv')
        if os.path.exists(progress_file):
            try:
                self.results = pd.read_csv(progress_file)
                self.current_image_index = len(self.results)
                self.status_label.value = f"恢复{respondent_name}的进度 ({self.current_image_index}/{len(self.current_respondent_images)})"
            except:
                self.results = pd.DataFrame(columns=['图片名'] + [f'问题{i+1}' for i in range(7)])
                self.status_label.value = f"开始{respondent_name}的评分"
                # 初始化所有滑块为4分
                for slider in self.question_widgets:
                    slider.value = 4
        else:
            self.results = pd.DataFrame(columns=['图片名'] + [f'问题{i+1}' for i in range(7)])
            self.status_label.value = f"开始{respondent_name}的评分"
            # 初始化所有滑块为4分
            for slider in self.question_widgets:
                slider.value = 4
        
        self.display_current_image()
        self.update_respondent_info()
    
    def create_widgets(self):
        """创建评分界面组件"""
        # 受访者信息输入区域
        self.respondent_name_input = widgets.Text(
            placeholder='请输入受访者名称或编号',
            description='受访者名称:',
            layout=widgets.Layout(width='300px')
        )
        
        self.respondent_info = widgets.HTML(
            value="<h3>等待开始测试...</h3>",
            layout=widgets.Layout(width='800px', border='1px solid blue', padding='10px', margin='10px 0')
        )
        
        # 图片显示区域 - 增大尺寸到900x600
        self.image_display = widgets.Output(
            layout=widgets.Layout(
                border='2px solid black', 
                width='920px',  # 增加宽度以适应更大图片
                height='620px',  # 增加高度
                margin='10px 0',
                display='flex',
                justify_content='center',
                align_items='center'
            )
        )
        
        # 问题滑块
        self.question_widgets = []
        for i, question in enumerate(self.questions):
            slider = widgets.IntSlider(
                value=4,
                min=1,
                max=7,
                step=1,
                description=question,
                continuous_update=False,
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='900px', margin='5px 0')  # 增加滑块宽度
            )
            slider.observe(self.on_slider_change, names='value')
            self.question_widgets.append(slider)
        
        # 控制按钮
        self.start_button = widgets.Button(
            description="开始新受访者", 
            button_style='success', 
            icon='play',
            layout=widgets.Layout(width='150px', margin='0 5px')
        )
        self.prev_button = widgets.Button(
            description="上一张", 
            button_style='info', 
            icon='arrow-left',
            layout=widgets.Layout(width='120px', margin='0 5px')
        )
        self.next_button = widgets.Button(
            description="下一张", 
            button_style='info', 
            icon='arrow-right',
            layout=widgets.Layout(width='120px', margin='0 5px')
        )
        self.finish_button = widgets.Button(
            description="完成当前受访者", 
            button_style='warning', 
            icon='check',
            layout=widgets.Layout(width='180px', margin='0 5px')
        )
        
        # 状态显示
        self.status_label = widgets.Label(value="系统就绪，请输入受访者名称或点击'开始新受访者'")
        self.progress_label = widgets.Label()
        
        # 布局
        self.questions_box = widgets.VBox(self.question_widgets, layout=widgets.Layout(margin='10px 0'))
        self.control_buttons = widgets.HBox(
            [self.prev_button, self.next_button, self.finish_button],
            layout=widgets.Layout(justify_content='center', margin='10px 0')
        )
        self.system_buttons = widgets.HBox(
            [self.respondent_name_input, self.start_button],
            layout=widgets.Layout(justify_content='center', margin='10px 0')
        )
        
        self.ui = widgets.VBox([
            self.system_buttons,
            self.respondent_info,
            self.image_display,
            self.questions_box,
            self.control_buttons,
            self.progress_label,
            self.status_label
        ], layout=widgets.Layout(width='950px', margin='20px'))  # 增加整体宽度
        
        # 绑定事件
        self.start_button.on_click(self.on_start_button_clicked)
        self.prev_button.on_click(self.on_prev_button_clicked)
        self.next_button.on_click(self.on_next_button_clicked)
        self.finish_button.on_click(self.on_finish_button_clicked)
        self.respondent_name_input.observe(self.on_respondent_name_change, names='value')
    
    def on_respondent_name_change(self, change):
        """受访者名称输入变化事件"""
        if change['new']:
            self.status_label.value = f"准备测试受访者: {change['new']}"
    
    def update_respondent_info(self):
        """更新受访者信息显示"""
        if self.current_respondent_id:
            completed_count = len([id for id, data in self.assignments.items() if data['status'] == '已完成'])
            total = self.num_respondents
            progress = (completed_count / total) * 100 if total > 0 else 0
            
            info_html = f"""
            <h3>当前测试信息</h3>
            <p><b>当前受访者:</b> {self.current_respondent_name} (ID: {self.current_respondent_id})</p>
            <p><b>总体进度:</b> {completed_count}/{total} ({progress:.1f}%)</p>
            <p><b>当前进度:</b> {self.current_image_index}/{len(self.current_respondent_images)}</p>
            <p><b>开始时间:</b> {datetime.now().strftime('%H:%M:%S')}</p>
            <p><b>状态:</b> {self.assignments[self.current_respondent_id]['status']}</p>
            """
            self.respondent_info.value = info_html
        else:
            # 显示可用受访者信息
            available_count = len([id for id, data in self.assignments.items() if data['status'] == '待测试'])
            completed_count = len([id for id, data in self.assignments.items() if data['status'] == '已完成'])
            
            info_html = f"""
            <h3>系统状态</h3>
            <p><b>总受访者:</b> {self.num_respondents}</p>
            <p><b>已完成:</b> {completed_count}</p>
            <p><b>待测试:</b> {available_count}</p>
            <p><b>请输入受访者名称或点击'开始新受访者'</b></p>
            """
            self.respondent_info.value = info_html
    
    def on_slider_change(self, change):
        """滑块值变化事件"""
        self.current_scores = [q.value for q in self.question_widgets]
    
    def on_start_button_clicked(self, b):
        """开始新受访者按钮点击事件"""
        # 如果输入了特定的受访者名称，尝试解析
        input_name = self.respondent_name_input.value.strip()
        if input_name:
            try:
                # 尝试从输入中提取受访者ID
                if input_name.startswith('受访者_'):
                    respondent_id = int(input_name.replace('受访者_', ''))
                else:
                    respondent_id = int(input_name)
                
                if 1 <= respondent_id <= self.num_respondents:
                    # 更新受访者名称
                    if input_name != f"受访者_{respondent_id}":
                        self.update_respondent_name(respondent_id, input_name)
                    
                    respondent_name = self.assignments[respondent_id]['respondent_name']
                    self.start_specific_respondent(respondent_id, respondent_name)
                    return
                else:
                    self.status_label.value = f"受访者ID必须在1-{self.num_respondents}之间"
                    return
            except ValueError:
                # 如果不是数字，创建自定义受访者（使用第一个可用ID）
                if self.available_respondents:
                    respondent_id = self.available_respondents[0]
                    # 更新受访者名称
                    self.update_respondent_name(respondent_id, input_name)
                    self.start_specific_respondent(respondent_id, input_name)
                else:
                    self.status_label.value = "没有可用的受访者ID，所有受访者已完成测试"
                return
        
        # 如果没有输入特定名称，开始下一个受访者
        if self.start_next_respondent():
            self.status_label.value = f"开始测试{self.current_respondent_name}"
        else:
            self.status_label.value = "所有受访者已完成测试！"
    
    def start_custom_respondent(self, respondent_name):
        """开始自定义受访者的测试"""
        # 为自定义受访者随机分配图片
        self.current_respondent_id = 999  # 自定义ID
        self.current_respondent_name = respondent_name
        self.current_respondent_images = random.sample(self.all_images, min(50, len(self.all_images)))
        self.current_image_index = 0
        self.current_scores = [4] * 7
        self.results = pd.DataFrame(columns=['图片名'] + [f'问题{i+1}' for i in range(7)])
        
        # 初始化所有滑块为4分
        for slider in self.question_widgets:
            slider.value = 4

        self.display_current_image()
        self.update_respondent_info()
        self.status_label.value = f"开始测试自定义受访者: {respondent_name}"
    
    def display_current_image(self):
        """显示当前图片"""
        if (self.current_respondent_id is None or 
            self.current_image_index >= len(self.current_respondent_images)):
            return
        
        with self.image_display:
            clear_output(wait=True)
            img_name = self.current_respondent_images[self.current_image_index]
            img_path = os.path.join(self.image_folder, img_name)
            
            try:
                img = Image.open(img_path)
                # 调整图片大小以适应显示区域（增大图片到900x600）
                max_width, max_height = 900, 600  # 增大显示尺寸
                img.thumbnail((max_width, max_height), Image.Resampling.LANCZOS)
                display(img)
            except Exception as e:
                error_msg = f"<div style='color: red; font-size: 16px;'>无法加载图片: {img_name}<br>错误: {e}</div>"
                display(widgets.HTML(value=error_msg))
        
        # 更新滑块值 - 始终设置为4分，除非已有保存的进度
        if (self.current_image_index < len(self.results) and 
            len(self.results) > 0 and 
            pd.notna(self.results.iloc[self.current_image_index, 1])):
            # 如果有保存的进度，使用保存的分数
            saved_scores = self.results.iloc[self.current_image_index, 1:].tolist()
            for i, score in enumerate(saved_scores):
                if pd.notna(score):
                    self.question_widgets[i].value = int(score)
        else:
            # 新图片或没有保存的进度，全部设置为4分
            for slider in self.question_widgets:
                slider.value = 4
        
        self.progress_label.value = f"图片 {self.current_image_index + 1}/{len(self.current_respondent_images)}: {img_name}"
        self.update_respondent_info()
    
    def save_current_progress(self):
        """保存当前进度"""
        if self.current_respondent_id is None:
            return
        
        self.current_scores = [q.value for q in self.question_widgets]
        
        if self.current_image_index < len(self.results):
            self.results.iloc[self.current_image_index, 1:] = self.current_scores
        else:
            new_row = [self.current_respondent_images[self.current_image_index]] + self.current_scores
            self.results.loc[len(self.results)] = new_row
        
        # 保存进度文件
        progress_file = os.path.join(self.results_dir, f'progress_respondent_{self.current_respondent_id}.csv')
        self.results.to_csv(progress_file, index=False, encoding='utf-8-sig')
    
    def on_prev_button_clicked(self, b):
        """上一张按钮"""
        if self.current_respondent_id is None:
            self.status_label.value = "请先开始测试"
            return
        
        self.save_current_progress()
        
        if self.current_image_index > 0:
            self.current_image_index -= 1
            self.display_current_image()
    
    def on_next_button_clicked(self, b):
        """下一张按钮"""
        if self.current_respondent_id is None:
            self.status_label.value = "请先开始测试"
            return
        
        self.save_current_progress()
        
        if self.current_image_index < len(self.current_respondent_images) - 1:
            self.current_image_index += 1
            self.display_current_image()
        else:
            self.status_label.value = "已是最后一张图片！请点击'完成当前受访者'保存结果"
    
    def on_finish_button_clicked(self, b):
        """完成当前受访者"""
        if self.current_respondent_id is None:
            return
        
        self.save_current_progress()
        
        # 生成结果文件名（受访者名称+日期）
        date_str = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"{self.current_respondent_name}_{date_str}.csv"
        output_path = os.path.join(self.results_dir, filename)
        
        # 保存最终结果
        self.results.to_csv(output_path, index=False, encoding='utf-8-sig')
        
        # 更新完成状态
        self.assignments[self.current_respondent_id]['status'] = '已完成'
        
        # 从可用列表中移除
        if self.current_respondent_id in self.available_respondents:
            self.available_respondents.remove(self.current_respondent_id)
        
        # 添加到完成列表
        self.completed_respondents[self.current_respondent_id] = self.current_respondent_name
        
        # 清理进度文件
        progress_file = os.path.join(self.results_dir, f'progress_respondent_{self.current_respondent_id}.csv')
        if os.path.exists(progress_file):
            os.remove(progress_file)
        
        # 更新临时记录文件（包含使用统计）
        self.save_records_to_file()
        
        self.status_label.value = f"{self.current_respondent_name} 完成！结果保存为: {filename}"
        
        # 重置状态，准备下一个受访者
        self.current_respondent_id = None
        self.current_respondent_name = ""
        self.respondent_name_input.value = ""
        self.progress_label.value = ""
        
        with self.image_display:
            clear_output()
        
        self.update_respondent_info()
    
    def show_interface(self):
        """显示评分界面"""
        display(self.ui)

# Launch the rating system from the repository root.
from pathlib import Path

image_folder = Path("example_images")
if not image_folder.is_dir():
    raise FileNotFoundError(
        "Create the 'example_images' folder and add the images to be rated before running the system."
    )

rating_system = MultiRespondentRatingSystem(
    str(image_folder),
    num_respondents=120,
    images_per_person=50,
)
rating_system.show_interface()